# Phase 1: SIR Measurement on GCR Baseline

**Goal:** Measure how permissive GCR's KG-Trie is using the Semantic Irrelevance Ratio (SIR).
This generates the baseline numbers for Chapter 4 that motivate DCA-Trie.

**What you'll get:**
- SIR table by hop depth (1/2/3/4)
- Average trie size per hop depth
- SIR vs. hop depth plot

**Runs on:** CPU only (no GPU needed). MiniLM runs fine on CPU.

**Prerequisites:** HuggingFace token (set HF_TOKEN below) to download the GCR tokenizer.

## 1. Colab Environment Setup

Run this cell to clone the repo and install dependencies.

In [ ]:
# --- Colab Setup ---
import sys, os, json, warnings, gc, time, textwrap, pprint, copy
import itertools, collections, random, math, re
import typing, pathlib, hashlib, functools
import numpy as np
from tqdm import tqdm
from collections import defaultdict

print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")

# Detect if running in Colab
IN_COLAB = 'google.colab' in sys.modules
print(f"Running in Colab: {IN_COLAB}")

if IN_COLAB:
    # Clone repo if not already present
    if not os.path.exists('dca-trie'):
        !git clone https://github.com/YOUR_USERNAME/dca-trie.git
    %cd dca-trie

    # Install dependencies
    # sentence-transformers and datasets are the main ones needed (CPU only)
    !pip install -q sentence-transformers datasets transformers scikit-learn marisa-trie

    # Install the package in dev mode so gcr/ and dca_trie/ are importable
    !pip install -e . --no-deps
else:
    # Running locally — assume poetry env is active
    print("Make sure you're in the poetry shell: source $(poetry env info --path)/bin/activate")

print("\nSetup complete.")

## 2. Set HuggingFace Token

The GCR tokenizer is a gated model — you need a HuggingFace token.
Get yours at https://huggingface.co/settings/tokens

Request access to the model: https://huggingface.co/rmanluo/GCR-Meta-Llama-3.1-8B-Instruct

In [ ]:
from huggingface_hub import login

HF_TOKEN = ""  # <-- SET YOUR TOKEN HERE

if not HF_TOKEN:
    # Fallback: try environment variable
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=True)
    print("HF_TOKEN configured.")
else:
    print("WARNING: No HF_TOKEN set. Tokenizer download may fail for gated models.")

## 3. Imports

Import the DCA-Trie modules and GCR baseline modules.

In [ ]:
# DCA-Trie modules (contribution)
from dca_trie.semantic_scorer import SemanticScorer
from dca_trie.sir_measurement import SIRMeasurer
from dca_trie.mid_resolver import MidResolver

# GCR baseline modules (vendored)
import gcr.src.utils as gcr_utils
from gcr.src.utils.graph_utils import build_graph, dfs, get_truth_paths
from gcr.src.trie import MarisaTrie

# HuggingFace
from datasets import load_dataset
from transformers import AutoTokenizer

%load_ext autoreload
%autoreload 2

print("All imports OK.")

## 4. Initialize Semantic Scorer

We use `all-MiniLM-L6-v2` (384-dim embeddings, ~80MB, CPU-friendly).

In [ ]:
scorer = SemanticScorer()

# Quick smoke test
test_score = scorer.score(
    "Barack Obama -> people.person.spouse -> Michelle Obama",
    "Who is the spouse of Barack Obama?"
)
print(f"Test score: {test_score:.4f}")
assert 0.0 < test_score < 1.0, "Score should be between 0 and 1"

## 5. Build MID Resolver

WebQSP graphs contain Freebase MIDs (`m.0c6q0`) instead of readable names (`Warsaw`).
MiniLM cannot score paths with raw MIDs, so we resolve them first.

In [ ]:
resolver = MidResolver(cache_path="data/mid_to_name.json")

# Load a small sample to build the resolver
sample = load_dataset("rmanluo/RoG-webqsp", split="test[:50]")
resolver.build_from_dataset(sample)

cov = resolver.coverage(sample)
print(f"MID coverage (from 50 questions): {cov['coverage_pct']}%")
print(f"  Resolved: {cov['resolved']} / {cov['total_mids']} MIDs")

# Show a few examples
print("\nSample resolutions:")
count = 0
for mid, name in resolver.mid_to_name.items():
    print(f"  {mid} -> {name}")
    count += 1
    if count >= 10:
        break

## 6. Patch GCR's path_to_string

We monkey-patch `gcr.src.utils.path_to_string` so that every path string
has its MIDs resolved to readable names before MiniLM sees them.

This must be done BEFORE we start building tries.

**Why patching?** Every module that calls `path_to_string()` (GCR's internals,
our V1TrieBuilder, this notebook) will automatically get resolved names.
No need to change any source code.

In [ ]:
_ORIG_PATH_TO_STR = gcr_utils.path_to_string

def _resolved_path_to_string(path):
    raw = _ORIG_PATH_TO_STR(path)
    return resolver.resolve_path(raw)

gcr_utils.path_to_string = _resolved_path_to_string

# Verify the patch works
from gcr.src.utils import path_to_string
print(f"path_to_string is patched: {path_to_string != _ORIG_PATH_TO_STR}")

# Test on sample data
g = build_graph(sample[0]["graph"])
test_paths = dfs(g, sample[0]["q_entity"], 2)
if test_paths:
    print(f"\nBefore patch (raw): {_ORIG_PATH_TO_STR(test_paths[0])}")
    print(f"After patch (resolved): {path_to_string(test_paths[0])}")

## 7. Load WebQSP Data

Load from HuggingFace and load the GCR tokenizer (needed for MarisaTrie construction).
The tokenizer requires downloading from HuggingFace but doesn't load the model (no GPU needed).

In [ ]:
NUM_QUESTIONS = 100  # Set to -1 for full test set

split = f"test[:{NUM_QUESTIONS}]" if NUM_QUESTIONS > 0 else "test"
dataset = load_dataset("rmanluo/RoG-webqsp", split=split)
questions = list(dataset)
print(f"Loaded {len(questions)} questions from WebQSP")
print(f"Sample question: {questions[0]['question']}")
print(f"Fields: {list(questions[0].keys())}")

# Load GCR tokenizer (model not needed)
print("\nLoading GCR tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct",
    trust_remote_code=True,
)
print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")

## 8. Build GCR-Style Tries

Replicates what GCR's `get_graph_index()` does:
1. Build directed graph from the question's triple list
2. DFS from q_entity up to 2 hops
3. Tokenize each path
4. Build a MarisaTrie

Because we patched `path_to_string`, all paths will have resolved MIDs.

In [ ]:
MAX_HOPS = 2

def build_gcr_trie(question_dict, tokenizer, max_hops=MAX_HOPS):
    """Replicate GCR's get_graph_index() to produce a MarisaTrie."""
    g = build_graph(question_dict["graph"])
    paths_list = dfs(g, question_dict["q_entity"], max_hops)

    if not paths_list:
        return None, []

    from gcr.src.utils import path_to_string as pts
    paths_list_str = [pts(p) for p in paths_list]
    tokenized_paths = tokenizer(
        paths_list_str, padding=False, add_special_tokens=False
    ).input_ids
    tokenized_path_list = [
        ids + [tokenizer.eos_token_id] for ids in tokenized_paths
    ]
    trie = MarisaTrie(tokenized_path_list, max_token_id=len(tokenizer) + 1)
    return trie, paths_list_str


# Test on first question
sample_trie, sample_paths = build_gcr_trie(questions[0], tokenizer)
print(f"Q: {questions[0]['question']}")
print(f"Paths in trie: {len(sample_trie)}")
print("\nFirst 3 paths (resolved):")
for p in sample_paths[:3]:
    print(f"  {p}")

## 9. Measure Baseline SIR

For each question, we:
1. Build the GCR trie (with MID resolution)
2. Measure overall SIR: `1 - max cos(path, question)`
3. Measure per-hop SIR (stratified by path depth)

**Expected trend:** SIR increases with hop depth, showing that GCR's
trie becomes increasingly permissive at deeper reasoning steps.

In [ ]:
measurer = SIRMeasurer(scorer, tokenizer)

per_question_results = []

for i, data in enumerate(tqdm(questions, desc="Measuring SIR")):
    trie, path_strs = build_gcr_trie(data, tokenizer)
    if trie is None:
        continue

    # Overall SIR
    sir_result = measurer.measure_from_trie(trie, data["question"])

    # Per-hop SIR
    hop_sir = measurer.measure_per_hop(trie, data["question"])

    per_question_results.append({
        "id": data.get("id", i),
        "question": data["question"],
        "num_paths": sir_result["num_paths"],
        "sir": sir_result["sir"],
        "max_sim": sir_result["max_similarity"],
        "avg_sim": sir_result["avg_similarity"],
        "hop_sir": hop_sir,
    })

print(f"\nMeasured SIR for {len(per_question_results)} questions")

# Show a few samples
for r in per_question_results[:5]:
    print(f"\n[{r['id']}] {r['question'][:70]}")
    print(f"  SIR: {r['sir']:.4f}  |  Paths: {r['num_paths']}  |  MaxSim: {r['max_sim']:.4f}")

## 10. Results Table (for Chapter 4)

Aggregate SIR values across all questions.

In [ ]:
print("=" * 72)
print(f"{'Metric':<30} {'Mean':<12} {'Min':<12} {'Max':<12}")
print("-" * 72)

all_sirs = [r["sir"] for r in per_question_results]
all_path_counts = [r["num_paths"] for r in per_question_results]
all_max_sims = [r["max_sim"] for r in per_question_results]

print(f"{'SIR':<30} {np.mean(all_sirs):<12.4f} {np.min(all_sirs):<12.4f} {np.max(all_sirs):<12.4f}")
print(f"{'Trie size (num paths)':<30} {np.mean(all_path_counts):<12.1f} {np.min(all_path_counts):<12.0f} {np.max(all_path_counts):<12.0f}")
print(f"{'Max similarity':<30} {np.mean(all_max_sims):<12.4f} {np.min(all_max_sims):<12.4f} {np.max(all_max_sims):<12.4f}")
print("=" * 72)

# Per-hop breakdown
print("\n" + "=" * 72)
print("SIR by Hop Depth")
print("=" * 72)
print(f"{'Hop':<8} {'Questions':<12} {'Avg Paths':<12} {'SIR':<12} {'MaxSim':<12}")
print("-" * 72)

all_hop_data = {h: {"sirs": [], "counts": []} for h in [1, 2, 3, 4]}
for r in per_question_results:
    for hop, hop_data in r["hop_sir"].items():
        if hop_data["num_paths"] > 0:
            all_hop_data[hop]["sirs"].append(hop_data["sir"])
            all_hop_data[hop]["counts"].append(hop_data["num_paths"])

for hop in [1, 2, 3, 4]:
    d = all_hop_data[hop]
    if d["sirs"]:
        print(f"{hop:<8} {len(d['sirs']):<12} {np.mean(d['counts']):<12.1f} {np.mean(d['sirs']):<12.4f} N/A")
    else:
        print(f"{hop:<8} {'N/A':<12} {'N/A':<12} {'N/A':<12} {'N/A':<12}")

## 11. SIR vs. Hop Depth Plot

This plot goes directly into your thesis Chapter 4.
It shows that GCR's trie becomes increasingly permissive at deeper hops.

In [ ]:
import matplotlib.pyplot as plt

hop_depths = []
mean_sirs = []
mean_trie_sizes = []

for hop in [1, 2, 3, 4]:
    d = all_hop_data[hop]
    if d["sirs"]:
        hop_depths.append(hop)
        mean_sirs.append(np.mean(d["sirs"]))
        mean_trie_sizes.append(np.mean(d["counts"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: SIR vs hop depth
ax1.plot(hop_depths, mean_sirs, "bo-", linewidth=2, markersize=8)
ax1.set_xlabel("Hop Depth", fontsize=12)
ax1.set_ylabel("Mean SIR", fontsize=12)
ax1.set_title("GCR Baseline: SIR Increases with Hop Depth", fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

# Right: Trie size vs hop depth
ax2.bar(hop_depths, mean_trie_sizes, color="orange", alpha=0.7)
ax2.set_xlabel("Hop Depth", fontsize=12)
ax2.set_ylabel("Avg Trie Size (paths)", fontsize=12)
ax2.set_title("GCR Baseline: Trie Size at Each Hop", fontsize=13)
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/gcr_baseline_sir.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to figures/gcr_baseline_sir.png")

## 12. Export Results for Thesis

Save aggregated results as JSON for your thesis tables.

In [ ]:
output = {
    "dataset": "RoG-webqsp",
    "num_questions": len(per_question_results),
    "model": "GCR-Meta-Llama-3.1-8B-Instruct",
    "index_path_length": MAX_HOPS,
    "mid_coverage_pct": cov["coverage_pct"],
    "overall": {
        "mean_sir": float(np.mean(all_sirs)),
        "median_sir": float(np.median(all_sirs)),
        "mean_trie_size": float(np.mean(all_path_counts)),
        "median_trie_size": float(np.median(all_path_counts)),
    },
    "per_hop": {},
    "per_question": per_question_results,
}

for hop in [1, 2, 3, 4]:
    d = all_hop_data[hop]
    if d["sirs"]:
        output["per_hop"][str(hop)] = {
            "mean_sir": float(np.mean(d["sirs"])),
            "mean_trie_size": float(np.mean(d["counts"])),
            "num_questions": len(d["sirs"]),
        }
    else:
        output["per_hop"][str(hop)] = {
            "mean_sir": None,
            "mean_trie_size": None,
            "num_questions": 0,
        }

os.makedirs("data", exist_ok=True)
with open("data/gcr_baseline_sir_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved to data/gcr_baseline_sir_results.json")
print(f"\nSummary for thesis:")
print(f"  Dataset: WebQSP ({output['num_questions']} questions)")
print(f"  Overall mean SIR: {output['overall']['mean_sir']:.4f}")
print(f"  Overall mean trie size: {output['overall']['mean_trie_size']:.1f}")
print(f"  Per-hop SIR:")
for k, v in output["per_hop"].items():
    if v["mean_sir"]:
        print(f"    Depth {k}: SIR={v['mean_sir']:.4f},  paths={v['mean_trie_size']:.1f},  n={v['num_questions']}")
    else:
        print(f"    Depth {k}: no data")

## 13. (Optional) CWQ Baseline SIR

Same measurement on ComplexWebQuestions for a more comprehensive baseline.

In [ ]:
# Uncomment to run on CWQ:
# cwq = load_dataset("rmanluo/RoG-cwq", split="test[:100]")
# ... same measurement pipeline ...
print("CWQ: uncomment and run after WebQSP validation.")

## Phase 1 Exit Criteria

- [ ] SIR measurement runs without errors
- [ ] Produces per-question and per-hop SIR values
- [ ] SIR shows clear upward trend with hop depth (motivates the project)
- [ ] Average trie size per step is reported
- [ ] Plot saved to `figures/gcr_baseline_sir.png`
- [ ] Results exported to `data/gcr_baseline_sir_results.json`
- [ ] MID resolver integrated (coverage reported)

**Expected finding:** SIR increases with hop depth, confirming that GCR's trie
becomes increasingly permissive for longer reasoning chains. This is the
empirical motivation for DCA-Trie.

Once confirmed, proceed to **Phase 2: DCA-Trie v1 Threshold Sweep**.